# Bronze: Region — Raw File Ingestion

Loads raw `region` CSV files from the source volume into the
bronze table **`saleslake_{env}.bronze_{env}.rawRegion`** using
Databricks `COPY INTO` for idempotent, exactly-once ingestion.

In [0]:
dbutils.widgets.dropdown(name="environment", defaultValue="dev",
                         choices=["dev","qa","prd"], label="select Environment")
env = dbutils.widgets.get("environment")
print(f"Environment: {env}")


In [0]:
tablName    = f"saleslake_{env}.bronze_{env}.rawRegion"
srcFileLoc  = f"s3://saleslake-748005667258-eu-north-1-an/saleslake/{env}/source_files/region/"
print(f"Target table : {tablName}")
print(f"Source files : {srcFileLoc}")


In [0]:
spark.sql(f"""
COPY INTO {tablName}
FROM (
    SELECT region_id, region_code, region_name, country, sub_region, regional_manager, created_date,
           current_timestamp() AS ingest_ts
    FROM "{srcFileLoc}"
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS  ('mergeSchema' = 'false')
""")

print(f"Bronze load complete for {tablName}")
spark.sql(f"SELECT COUNT(*) AS row_count FROM {tablName}").display()
